# 🏓 TTNet — نوتبوك التدريب الكامل
### Complete Training Notebook with Frame Visualization

---

## ما يشمله هذا النوتبوك | What this notebook covers

| القسم | الوصف |
|---|---|
| ⚙️ الإعداد | فحص GPU، تثبيت المكتبات، ضبط المسارات |
| 🎞️ زيادة الفريمات | رفع `num_frames_sequence` من 9 إلى **15** فريم |
| 🔍 عرض الفريمات | عرض كل فريم في التسلسل + تمييز الفريم المحوري |
| 📊 أهمية الفريمات | تحليل Gradient Saliency لمعرفة على أي فريم يركز النموذج |
| 🏋️ المرحلة الأولى | تدريب Global Ball Detection + Segmentation |
| 🏋️ المرحلة الثانية | إضافة Local Ball Detection + Event Spotting |
| 🏋️ المرحلة الثالثة | Fine-tuning كامل لجميع المهام |
| 📈 التقييم | منحنيات الخسارة + تقييم نهائي |

---
### بنية التسلسل الزمني | Frame Sequence Structure
```
مع 15 فريم (num_frames_sequence=15):

  F-7  F-6  F-5  F-4  F-3  F-2  F-1  [F0]  F+1  F+2  F+3  F+4  F+5  F+6  [F+7]
   ↑                                    ↑                                     ↑
أقدم فريم                          حدث (bounce/net)                    أحدث فريم
                                                                     ← موضع الكرة هنا
```

## 1️⃣ إعداد البيئة | Environment Setup

In [ ]:
import torch
import platform
import sys
import os

print('='*60)
print('🖥️  معلومات البيئة | Environment Info')
print('='*60)
print(f'Python version   : {sys.version.split()[0]}')
print(f'PyTorch version  : {torch.__version__}')
print(f'Platform         : {platform.system()} {platform.release()}')
print()

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f'✅ GPU متوفر | GPU available: {n_gpus} device(s)')
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f'   GPU {i}: {props.name} | VRAM: {props.total_memory // 1024**3} GB')
    DEVICE = 'cuda:0'
    GPU_IDX = 0
else:
    print('⚠️  لا يوجد GPU — سيعمل على CPU (سيكون أبطأ)')
    DEVICE = 'cpu'
    GPU_IDX = None

print(f'\nالجهاز المستخدم | Device: {DEVICE}')

In [ ]:
# تثبيت المكتبات المطلوبة | Install required packages
import subprocess

packages = [
    'easydict',
    'scikit-learn',
    'PyTurboJPEG',
    'tqdm',
    'tensorboard',
    'matplotlib',
    'opencv-python',
    'numpy',
]

print('📦 تثبيت/التحقق من المكتبات...')
for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
        capture_output=True, text=True
    )
    status = '✅' if result.returncode == 0 else '❌'
    print(f'  {status} {pkg}')

print('\n✅ جاهز!')

## 2️⃣ ضبط المسارات والإعدادات | Paths & Configuration

In [ ]:
import os

# ═══════════════════════════════════════════════════
# ⚙️  اضبط هذه المسارات حسب جهازك | Set your paths
# ═══════════════════════════════════════════════════

# المجلد الجذر للمشروع | Root of the project repo
REPO_ROOT = os.path.abspath(os.path.dirname('__file__') if '__file__' in dir() else os.getcwd())
# إذا كنت على Colab وعندك Drive | If on Colab with Drive:
# REPO_ROOT = '/content/drive/MyDrive/TTNet-Real-time-Analysis-System-for-Table-Tennis-Pytorch'

WORKING_DIR = REPO_ROOT          # الجذر الذي يحتوي dataset/ و checkpoints/
SRC_DIR     = os.path.join(REPO_ROOT, 'src')
DATASET_DIR = os.path.join(WORKING_DIR, 'dataset')
CKPT_DIR    = os.path.join(WORKING_DIR, 'checkpoints')
LOGS_DIR    = os.path.join(WORKING_DIR, 'logs')

# ═══════════════════════════════════════════════════
# 🎞️  عدد الفريمات في التسلسل | Frames per sequence
# الورقة البحثية ذكرت 25، الكود الأصلي يستخدم 9
# نرفعه إلى 15 لتحسين الدقة مع الحفاظ على الذاكرة
# ═══════════════════════════════════════════════════
NUM_FRAMES = 15    # يجب أن يكون عدداً فردياً (odd number)

# ═══════════════════════════════════════════════════
# 🏋️  إعدادات التدريب | Training settings
# ═══════════════════════════════════════════════════
BATCH_SIZE   = 8
NUM_WORKERS  = 4
NUM_EPOCHS_1 = 30    # Phase 1
NUM_EPOCHS_2 = 30    # Phase 2
NUM_EPOCHS_3 = 30    # Phase 3

# إضافة مجلد src إلى مسار Python
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print('📁 المسارات | Paths:')
print(f'  REPO_ROOT   : {REPO_ROOT}')
print(f'  SRC_DIR     : {SRC_DIR}')
print(f'  DATASET_DIR : {DATASET_DIR}')
print(f'  CKPT_DIR    : {CKPT_DIR}')
print()
print(f'🎞️  NUM_FRAMES  : {NUM_FRAMES} (↑ من 9 الأصلية)')
print(f'📐 نافذة الحدث : ±{(NUM_FRAMES-1)//2} فريم حول لحظة الحدث')

# التحقق من وجود مجلد البيانات
if os.path.isdir(DATASET_DIR):
    print(f'\n✅ مجلد البيانات موجود')
    contents = os.listdir(DATASET_DIR)
    print(f'   المحتويات: {contents}')
else:
    print(f'\n⚠️  مجلد البيانات غير موجود: {DATASET_DIR}')
    print('   قم بتشغيل prepare_dataset/ أولاً')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# دالة مساعدة: إنشاء configs مع تجاوز الإعدادات الافتراضية
# Helper: create configs while overriding defaults cleanly
# ═══════════════════════════════════════════════════════════════

def make_configs(extra_args=None):
    """Create configs with proper working dir and num_frames override."""
    base_args = [
        'notebook',
        '--working-dir', WORKING_DIR,
        '--num_workers', str(NUM_WORKERS),
        '--batch_size', str(BATCH_SIZE),
    ]
    if GPU_IDX is not None:
        base_args += ['--gpu_idx', str(GPU_IDX)]
    else:
        base_args += ['--no_cuda']

    if extra_args:
        base_args += extra_args

    original_argv = sys.argv.copy()
    sys.argv = base_args

    from config.config import parse_configs
    configs = parse_configs()

    sys.argv = original_argv

    # ← الجزء المهم: تجاوز عدد الفريمات
    configs.num_frames_sequence = NUM_FRAMES
    configs.smooth_labelling = True

    return configs


# اختبار سريع
test_cfg = make_configs()
print(f'✅ num_frames_sequence = {test_cfg.num_frames_sequence}')
print(f'   نافذة من كل طرف   = {(test_cfg.num_frames_sequence-1)//2} فريم')
print(f'   input_size         = {test_cfg.input_size}')
print(f'   الجهاز             = {test_cfg.device}')

## 3️⃣ استكشاف البيانات | Dataset Exploration

In [ ]:
from collections import Counter
from data_process.ttnet_data_utils import get_events_infor, train_val_data_separation

configs = make_configs(['--smooth-labelling', '--no-val'])

print('📊 تحميل معلومات التسلسلات...')
train_ei, val_ei, train_lbl, val_lbl = train_val_data_separation(configs)

print()
print('='*55)
print(f'  إجمالي التسلسلات التدريبية  : {len(train_ei):,}')
if val_ei:
    print(f'  إجمالي تسلسلات التحقق      : {len(val_ei):,}')
print()

label_names = {0: '🏀 Bounce', 1: '🕸️  Net', 2: '⬜ Empty'}
cnt = Counter(train_lbl)
print('  توزيع الفئات | Class distribution:')
total = sum(cnt.values())
for cls_id, name in label_names.items():
    n = cnt.get(cls_id, 0)
    bar = '█' * int(30 * n / total)
    print(f'  {name:<12} {n:>5,}  {bar}')

# معلومات عن التسلسل
sample = train_ei[0]
img_paths, ball_pos, events, seg_path = sample
print()
print(f'  فريمات في كل تسلسل : {len(img_paths)}')
print(f'  موضع الكرة (org)   : x={ball_pos[0]}, y={ball_pos[1]}')
print(f'  label الحدث        : bounce={events[0]:.3f}, net={events[1]:.3f}')
print(f'  مسار التجزئة       : {os.path.basename(seg_path)}')

## 4️⃣ عرض الفريمات | Frame Sequence Visualization

كل عينة تدريبية تتكون من **15 فريم متتالي** حول لحظة الحدث:
- **F0** (الفريم المركزي) = لحظة الحدث (bounce أو net hit)
- **F+7** (آخر فريم) = الفريم الذي تُحدَّد فيه موضع الكرة
- الفريمات F-7 إلى F-1 = سياق **قبل** الحدث
- الفريمات F+1 إلى F+6 = سياق **بعد** الحدث

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

def visualize_frame_sequence(events_infor, sample_idx=0, num_frames=NUM_FRAMES):
    """عرض كل فريم في التسلسل مع تمييز الفريمات المهمة"""
    img_paths, ball_pos, events, seg_path = events_infor[sample_idx]

    half = (num_frames - 1) // 2
    event_frame_idx = half        # الفريم المركزي = F0
    ball_frame_idx  = num_frames - 1  # آخر فريم = F+half

    # تحديد نوع الحدث
    if events[0] > 0.5:
        event_type = f'🏀 Bounce (conf={events[0]:.2f})'
    elif events[1] > 0.5:
        event_type = f'🕸️  Net Hit (conf={events[1]:.2f})'
    else:
        event_type = '⬜ Empty (لا حدث)'

    # حساب عدد الصفوف والأعمدة
    ncols = min(num_frames, 5)
    nrows = (num_frames + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 2.5 + 1))
    axes = np.array(axes).ravel()

    for i, img_path in enumerate(img_paths):
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (320, 128))

        axes[i].imshow(img)
        axes[i].axis('off')

        # تسمية الفريم بالنسبة للحدث
        rel = i - event_frame_idx
        rel_str = f'F{rel:+d}' if rel != 0 else 'F0'

        # ألوان الإطار حسب أهمية الفريم
        if i == event_frame_idx:
            color = '#FF6B35'   # برتقالي ← فريم الحدث
            label = f'{rel_str}\n⚡ حدث'
        elif i == ball_frame_idx:
            color = '#E63946'   # أحمر ← فريم الكرة
            label = f'{rel_str}\n🏓 كرة'
            # رسم موضع الكرة
            cx, cy = int(ball_pos[0] / 6), int(ball_pos[1] / 8.4375)
            cx = max(3, min(317, cx))
            cy = max(3, min(125, cy))
            circle = plt.Circle((cx, cy), 5, color='lime', linewidth=2, fill=False)
            axes[i].add_patch(circle)
        else:
            color = '#457B9D'   # أزرق ← فريمات السياق
            label = rel_str

        for spine in axes[i].spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(3)

        axes[i].set_title(label, fontsize=9, color=color, fontweight='bold')

    # إخفاء المحاور الزائدة
    for j in range(len(img_paths), len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(
        f'عينة #{sample_idx} | {event_type}\n'
        f'{num_frames} فريم متتالي  |  🟠=حدث (F0)  🔴=كرة (F+{half})  🔵=سياق',
        fontsize=12, y=1.02
    )
    plt.tight_layout()
    plt.show()


# عرض 3 عينات مختلفة
for idx in [0, 50, 100]:
    try:
        visualize_frame_sequence(train_ei, sample_idx=idx)
    except Exception as e:
        print(f'تعذر عرض العينة {idx}: {e}')

In [ ]:
# ══════════════════════════════════════════════
# خط زمني للفريمات | Frame Timeline Diagram
# ══════════════════════════════════════════════

def plot_frame_timeline(num_frames=NUM_FRAMES):
    half = (num_frames - 1) // 2
    frames = list(range(-half, half + 1))

    fig, ax = plt.subplots(figsize=(14, 3))
    ax.set_xlim(-half - 1, half + 2)
    ax.set_ylim(-0.5, 1.5)
    ax.axis('off')

    # خط الزمن
    ax.annotate('', xy=(half + 1.5, 0.5), xytext=(-half - 0.5, 0.5),
                arrowprops=dict(arrowstyle='->', color='gray', lw=2))
    ax.text(half + 1.8, 0.5, 'الزمن →', va='center', color='gray')

    for i, f in enumerate(frames):
        if f == 0:
            color = '#FF6B35'
            label = f'F0\n⚡حدث'
            size = 0.55
        elif i == num_frames - 1:
            color = '#E63946'
            label = f'F+{half}\n🏓كرة'
            size = 0.55
        elif f < 0:
            color = '#A8DADC'
            label = f'F{f:+d}'
            size = 0.4
        else:
            color = '#457B9D'
            label = f'F+{f}'
            size = 0.4

        rect = plt.Rectangle((f - size/2, 0.5 - size/2), size, size,
                               color=color, ec='white', lw=1.5, zorder=3)
        ax.add_patch(rect)
        ax.text(f, 0.5, label, ha='center', va='center',
                fontsize=7 if abs(f) > 3 else 8,
                color='white', fontweight='bold', zorder=4)

    # مفتاح الألوان
    legend = [
        mpatches.Patch(color='#A8DADC', label=f'سياق قبل الحدث ({half} فريم)'),
        mpatches.Patch(color='#FF6B35', label='F0: الحدث (bounce/net)'),
        mpatches.Patch(color='#457B9D', label=f'سياق بعد الحدث ({half-1} فريم)'),
        mpatches.Patch(color='#E63946', label=f'F+{half}: موضع الكرة + التجزئة'),
    ]
    ax.legend(handles=legend, loc='upper center', bbox_to_anchor=(0.5, 1.35),
              ncol=2, fontsize=9, framealpha=0.9)

    ax.set_title(f'الخط الزمني للفريمات | num_frames_sequence = {num_frames}', pad=45)
    plt.tight_layout()
    plt.show()

plot_frame_timeline(NUM_FRAMES)

## 5️⃣ تحليل أهمية الفريمات | Frame Importance Analysis

### كيف يعمل هذا التحليل؟
نستخدم **Gradient Saliency** لمعرفة على أي فريم يعتمد النموذج:
1. نمرر عينة للنموذج (Forward Pass)
2. نحسب gradient الخسارة بالنسبة لكل بكسل في المدخل
3. نجمع قيم الـ gradient لكل 3 قنوات (RGB) خاصة بكل فريم
4. الفريم الذي له gradient أكبر ← النموذج يركز عليه أكثر

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt


def compute_frame_importance(model, input_tensor, num_frames, device):
    """
    Gradient Saliency per frame.
    input_tensor: (1, num_frames*3, H, W)  — على الـ device المناسب
    Returns: numpy array (num_frames,) — أهمية نسبية لكل فريم
    """
    model.eval()
    inner = model.module if hasattr(model, 'module') else model

    inp = input_tensor.clone().detach().float().to(device)
    inp.requires_grad_(True)

    with torch.enable_grad():
        normalized = inner._TTNet__normalize__(inp)
        pred_global, _, _, _, _, _ = inner.ball_global_stage(normalized)
        # backprop على أقوى استجابة
        pred_global.max().backward()

    grads = inp.grad[0].cpu()   # (num_frames*3, H, W)

    importances = []
    for f in range(num_frames):
        fg = grads[f * 3: (f + 1) * 3]   # (3, H, W)
        importances.append(fg.abs().mean().item())

    imp = np.array(importances)
    return imp / (imp.sum() + 1e-8)


def plot_frame_importance(importances_dict, num_frames=NUM_FRAMES, title=''):
    """
    importances_dict: {'Phase 1': array, 'Phase 2': array, ...}
    """
    half = (num_frames - 1) // 2
    x = np.arange(num_frames)
    labels = [f'F{i - half:+d}' if i != half else 'F0\n(حدث)' for i in range(num_frames)]
    labels[-1] = f'F+{half}\n(كرة)'

    n_phases = len(importances_dict)
    fig, axes = plt.subplots(1, n_phases, figsize=(7 * n_phases, 4.5), sharey=True)
    if n_phases == 1:
        axes = [axes]

    colors = ['#A8DADC'] * num_frames
    colors[half] = '#FF6B35'         # الفريم المركزي (حدث)
    colors[num_frames - 1] = '#E63946'  # آخر فريم (كرة)

    for ax, (phase_name, imp) in zip(axes, importances_dict.items()):
        bars = ax.bar(x, imp, color=colors, edgecolor='white', linewidth=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=8)
        ax.set_title(phase_name, fontsize=12, fontweight='bold')
        ax.set_ylabel('الأهمية النسبية' if ax == axes[0] else '')
        ax.set_xlabel('الفريم (بالنسبة للحدث)')

        # إضافة قيمة فوق كل عمود
        for bar, v in zip(bars, imp):
            ax.text(bar.get_x() + bar.get_width() / 2, v + 0.002,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=6.5)

        ax.axvline(half, color='#FF6B35', linestyle='--', alpha=0.5, linewidth=1.5)
        ax.axvline(num_frames - 1, color='#E63946', linestyle='--', alpha=0.5, linewidth=1.5)
        ax.set_ylim(0, max(imp) * 1.25)
        ax.grid(axis='y', alpha=0.3)

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#FF6B35', label='F0: فريم الحدث'),
        Patch(facecolor='#E63946', label=f'F+{half}: فريم الكرة'),
        Patch(facecolor='#A8DADC', label='فريمات السياق'),
    ]
    fig.legend(handles=legend_elements, loc='upper center',
               bbox_to_anchor=(0.5, 1.05), ncol=3, fontsize=10)
    fig.suptitle(title, fontsize=14, y=1.12)
    plt.tight_layout()
    plt.show()


print('✅ دوال تحليل أهمية الفريمات جاهزة')

## 6️⃣ المرحلة الأولى | Phase 1: Global Ball Detection + Segmentation

في هذه المرحلة يتعلم النموذج:
- **Global Ball Detection**: تحديد موضع الكرة في الصورة كاملة
- **Segmentation**: تقسيم الصورة إلى طاولة / لاعبين / خلفية

**لا يوجد**: Local stage أو Event spotting في هذه المرحلة

In [ ]:
import subprocess
import sys
import os
import time

phase1_ckpt = os.path.join(CKPT_DIR, 'ttnet_1st_phase',
                            f'ttnet_1st_phase_epoch_{NUM_EPOCHS_1}.pth')

if os.path.isfile(phase1_ckpt):
    print(f'✅ Checkpoint المرحلة الأولى موجود: {phase1_ckpt}')
    print('   تخطي التدريب — لإعادة التدريب احذف الملف أو غيّر اسم saved_fn')
else:
    print('🏋️  بدء تدريب المرحلة الأولى...')
    print(f'   Epochs: {NUM_EPOCHS_1}  |  Batch: {BATCH_SIZE}  |  Frames: {NUM_FRAMES}')

    cmd = [
        sys.executable, 'main.py',
        '--working-dir', WORKING_DIR,
        '--saved_fn', 'ttnet_1st_phase',
        '--no-val',
        '--batch_size', str(BATCH_SIZE),
        '--num_workers', str(NUM_WORKERS),
        '--num_epochs', str(NUM_EPOCHS_1),
        '--lr', '0.001',
        '--lr_type', 'step_lr',
        '--lr_step_size', '10',
        '--lr_factor', '0.1',
        '--global_weight', '5.0',
        '--seg_weight', '1.0',
        '--no_local',
        '--no_event',
        '--smooth-labelling',
        '--print_freq', '20',
    ]

    if GPU_IDX is not None:
        cmd += ['--gpu_idx', str(GPU_IDX)]
    else:
        cmd += ['--no_cuda']

    print('\nالأمر المنفذ:')
    print(' '.join(cmd[2:]))
    print()

    start = time.time()
    proc = subprocess.Popen(
        cmd,
        cwd=SRC_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    for line in proc.stdout:
        print(line, end='')

    proc.wait()
    elapsed = time.time() - start

    if proc.returncode == 0:
        print(f'\n✅ المرحلة الأولى انتهت في {elapsed/60:.1f} دقيقة')
    else:
        print(f'\n❌ خطأ في المرحلة الأولى (exit code {proc.returncode})')

In [ ]:
# ══════════════════════════════════════════════════════
# تحليل أهمية الفريمات بعد المرحلة الأولى
# Frame importance after Phase 1
# ══════════════════════════════════════════════════════

from models.model_utils import create_model
from data_process.ttnet_dataloader import create_train_val_dataloader

def load_model_from_checkpoint(ckpt_path, phase_args):
    """تحميل النموذج من checkpoint"""
    cfg = make_configs(phase_args)
    model = create_model(cfg)
    model = model.to(cfg.device)

    if os.path.isfile(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=cfg.device)
        state = ckpt.get('state_dict', ckpt)
        # إزالة بادئة 'model.' إن وجدت
        new_state = {k.replace('model.', ''): v for k, v in state.items()}
        if hasattr(model, 'module'):
            model.module.ttnet.load_state_dict(new_state, strict=False)
        else:
            try:
                model.ttnet.load_state_dict(new_state, strict=False)
            except Exception:
                model.load_state_dict(new_state, strict=False)
        print(f'✅ تم تحميل الـ checkpoint: {os.path.basename(ckpt_path)}')
    else:
        print(f'⚠️  checkpoint غير موجود: {ckpt_path}  — سيتم استخدام أوزان عشوائية')

    return model, cfg


def get_sample_tensor(cfg, num_samples=1):
    """الحصول على عينة من بيانات التدريب"""
    loader, _, _ = create_train_val_dataloader(cfg)
    batch = next(iter(loader))
    resized_imgs = batch[0][:num_samples]   # (N, C, H, W)
    return resized_imgs


# تحليل أهمية الفريمات
p1_args = [
    '--no-val', '--no_local', '--no_event', '--smooth-labelling',
    '--saved_fn', 'ttnet_1st_phase',
    '--global_weight', '5.0', '--seg_weight', '1.0',
]

model_p1, cfg_p1 = load_model_from_checkpoint(phase1_ckpt, p1_args)
sample_tensor_p1 = get_sample_tensor(cfg_p1)
imp_p1 = compute_frame_importance(model_p1, sample_tensor_p1, NUM_FRAMES, cfg_p1.device)

print('\n📊 أهمية كل فريم بعد المرحلة الأولى:')
half = (NUM_FRAMES - 1) // 2
for i, v in enumerate(imp_p1):
    rel = i - half
    marker = '← فريم الحدث ⚡' if i == half else ('← فريم الكرة 🏓' if i == NUM_FRAMES-1 else '')
    bar = '█' * int(v * 200)
    print(f'  F{rel:+3d}: {v:.4f}  {bar} {marker}')

plot_frame_importance({'Phase 1 — Global + Seg': imp_p1},
                      title='أهمية الفريمات | Frame Importance (Phase 1)')

## 7️⃣ المرحلة الثانية | Phase 2: Local Ball Detection + Event Spotting

في هذه المرحلة:
- **تجميد** الـ global stage و segmentation
- تدريب **Local Ball Detection**: تكبير منطقة الكرة والتنبؤ بموضعها بدقة أعلى
- تدريب **Event Spotting**: التعرف على bounce و net hit

In [ ]:
phase2_ckpt = os.path.join(CKPT_DIR, 'ttnet_2nd_phase',
                            f'ttnet_2nd_phase_epoch_{NUM_EPOCHS_2}.pth')

if os.path.isfile(phase2_ckpt):
    print(f'✅ Checkpoint المرحلة الثانية موجود: {phase2_ckpt}')
else:
    if not os.path.isfile(phase1_ckpt):
        print('❌ يجب أولاً إكمال المرحلة الأولى')
    else:
        print('🏋️  بدء تدريب المرحلة الثانية...')

        cmd = [
            sys.executable, 'main.py',
            '--working-dir', WORKING_DIR,
            '--saved_fn', 'ttnet_2nd_phase',
            '--no-val',
            '--batch_size', str(BATCH_SIZE),
            '--num_workers', str(NUM_WORKERS),
            '--num_epochs', str(NUM_EPOCHS_2),
            '--lr', '0.001',
            '--lr_type', 'step_lr',
            '--lr_step_size', '10',
            '--lr_factor', '0.1',
            '--global_weight', '0.0',
            '--seg_weight', '0.0',
            '--event_weight', '2.0',
            '--local_weight', '1.0',
            '--pretrained_path', phase1_ckpt,
            '--overwrite_global_2_local',
            '--freeze_seg',
            '--freeze_global',
            '--smooth-labelling',
            '--print_freq', '20',
        ]

        if GPU_IDX is not None:
            cmd += ['--gpu_idx', str(GPU_IDX)]
        else:
            cmd += ['--no_cuda']

        start = time.time()
        proc = subprocess.Popen(cmd, cwd=SRC_DIR,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='')
        proc.wait()
        elapsed = time.time() - start

        if proc.returncode == 0:
            print(f'\n✅ المرحلة الثانية انتهت في {elapsed/60:.1f} دقيقة')
        else:
            print(f'\n❌ خطأ في المرحلة الثانية')

In [ ]:
# أهمية الفريمات بعد المرحلة الثانية
p2_args = [
    '--no-val', '--smooth-labelling',
    '--saved_fn', 'ttnet_2nd_phase',
    '--global_weight', '0.0', '--seg_weight', '0.0',
    '--event_weight', '2.0', '--local_weight', '1.0',
]

model_p2, cfg_p2 = load_model_from_checkpoint(phase2_ckpt, p2_args)
sample_tensor_p2 = get_sample_tensor(cfg_p2)
imp_p2 = compute_frame_importance(model_p2, sample_tensor_p2, NUM_FRAMES, cfg_p2.device)

# مقارنة المرحلتين
plot_frame_importance(
    {
        'Phase 1 — Global + Seg': imp_p1,
        'Phase 2 — Local + Event': imp_p2,
    },
    title='مقارنة أهمية الفريمات | Phase 1 vs Phase 2'
)

## 8️⃣ المرحلة الثالثة | Phase 3: Full Fine-Tuning

المرحلة الأخيرة: تدريب **جميع** وحدات النموذج معاً بـ learning rate منخفض
- Global + Local Ball Detection
- Event Spotting
- Segmentation
- **لا تجميد** لأي وحدة

In [ ]:
phase3_ckpt = os.path.join(CKPT_DIR, 'ttnet_3rd_phase',
                            f'ttnet_3rd_phase_epoch_{NUM_EPOCHS_3}.pth')

if os.path.isfile(phase3_ckpt):
    print(f'✅ Checkpoint المرحلة الثالثة موجود: {phase3_ckpt}')
else:
    if not os.path.isfile(phase2_ckpt):
        print('❌ يجب أولاً إكمال المرحلة الثانية')
    else:
        print('🏋️  بدء تدريب المرحلة الثالثة (Fine-tuning)...')

        cmd = [
            sys.executable, 'main.py',
            '--working-dir', WORKING_DIR,
            '--saved_fn', 'ttnet_3rd_phase',
            '--no-val',
            '--batch_size', str(BATCH_SIZE),
            '--num_workers', str(NUM_WORKERS),
            '--num_epochs', str(NUM_EPOCHS_3),
            '--lr', '0.0001',
            '--lr_type', 'step_lr',
            '--lr_step_size', '10',
            '--lr_factor', '0.2',
            '--global_weight', '1.0',
            '--seg_weight', '1.0',
            '--event_weight', '1.0',
            '--local_weight', '1.0',
            '--pretrained_path', phase2_ckpt,
            '--smooth-labelling',
            '--print_freq', '20',
        ]

        if GPU_IDX is not None:
            cmd += ['--gpu_idx', str(GPU_IDX)]
        else:
            cmd += ['--no_cuda']

        start = time.time()
        proc = subprocess.Popen(cmd, cwd=SRC_DIR,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='')
        proc.wait()
        elapsed = time.time() - start

        if proc.returncode == 0:
            print(f'\n✅ المرحلة الثالثة انتهت في {elapsed/60:.1f} دقيقة')
        else:
            print(f'\n❌ خطأ في المرحلة الثالثة')

In [ ]:
# أهمية الفريمات بعد المرحلة الثالثة
p3_args = [
    '--no-val', '--smooth-labelling',
    '--saved_fn', 'ttnet_3rd_phase',
]

model_p3, cfg_p3 = load_model_from_checkpoint(phase3_ckpt, p3_args)
sample_tensor_p3 = get_sample_tensor(cfg_p3)
imp_p3 = compute_frame_importance(model_p3, sample_tensor_p3, NUM_FRAMES, cfg_p3.device)

# مقارنة جميع المراحل
plot_frame_importance(
    {
        'Phase 1\nGlobal + Seg': imp_p1,
        'Phase 2\nLocal + Event': imp_p2,
        'Phase 3\nFull Fine-tune': imp_p3,
    },
    title='تطور أهمية الفريمات عبر مراحل التدريب | Frame Importance Evolution'
)

## 9️⃣ تحليل متقدم لأهمية الفريمات | Advanced Frame Attention Analysis

In [ ]:
# ══════════════════════════════════════════════════════════════
# Heatmap مكاني لكل فريم (Spatial Saliency per Frame)
# يُظهر أي منطقة من الصورة يركز عليها النموذج في كل فريم
# ══════════════════════════════════════════════════════════════

def compute_spatial_saliency(model, input_tensor, num_frames, device):
    """
    Returns: (num_frames, H, W) — saliency map per frame
    """
    model.eval()
    inner = model.module if hasattr(model, 'module') else model

    inp = input_tensor.clone().detach().float().to(device)
    inp.requires_grad_(True)

    with torch.enable_grad():
        normalized = inner._TTNet__normalize__(inp)
        pred_global, _, _, _, _, _ = inner.ball_global_stage(normalized)
        pred_global.max().backward()

    grads = inp.grad[0].cpu().numpy()    # (num_frames*3, H, W)
    H, W = grads.shape[1], grads.shape[2]

    saliency = np.zeros((num_frames, H, W))
    for f in range(num_frames):
        fg = np.abs(grads[f * 3: (f + 1) * 3])   # (3, H, W)
        saliency[f] = fg.mean(axis=0)             # (H, W)

    # Normalize per-frame
    for f in range(num_frames):
        mx = saliency[f].max()
        if mx > 0:
            saliency[f] /= mx

    return saliency


def plot_spatial_saliency(model, sample_tensor, img_paths, num_frames, device, title=''):
    """
    عرض الصورة الأصلية + خريطة الأهمية المكانية لكل فريم
    """
    saliency = compute_spatial_saliency(model, sample_tensor, num_frames, device)
    half = (num_frames - 1) // 2

    ncols = min(num_frames, 5)
    nrows = (num_frames + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows * 2, ncols, figsize=(ncols * 3, nrows * 4))
    axes = np.array(axes).reshape(nrows * 2, ncols)

    for i in range(num_frames):
        row_img = (i // ncols) * 2
        row_sal = row_img + 1
        col = i % ncols

        # الصورة الأصلية
        img = cv2.cvtColor(cv2.imread(img_paths[i]), cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (320, 128))
        axes[row_img, col].imshow(img)
        axes[row_img, col].axis('off')

        rel = i - half
        rel_str = f'F{rel:+d}' if rel != 0 else 'F0 (حدث)'
        if i == num_frames - 1:
            rel_str = f'F+{half} (كرة)'
        axes[row_img, col].set_title(rel_str, fontsize=8)

        # خريطة الأهمية
        axes[row_sal, col].imshow(saliency[i], cmap='hot', vmin=0, vmax=1)
        axes[row_sal, col].axis('off')

    # إخفاء المحاور الفارغة
    for i in range(num_frames, nrows * ncols):
        r_img = (i // ncols) * 2
        c = i % ncols
        if r_img < nrows * 2:
            axes[r_img, c].set_visible(False)
            if r_img + 1 < nrows * 2:
                axes[r_img + 1, c].set_visible(False)

    fig.suptitle(
        f'{title}\nالصف العلوي: الفريم الأصلي  |  الصف السفلي: خريطة الأهمية (أحمر = تركيز أكبر)',
        fontsize=11
    )
    plt.tight_layout()
    plt.show()


# عرض الخريطة المكانية للمرحلة الثالثة
try:
    sample_ei = train_ei[0]
    img_paths_sample = sample_ei[0]
    plot_spatial_saliency(
        model_p3, sample_tensor_p3, img_paths_sample,
        NUM_FRAMES, cfg_p3.device,
        title='المرحلة 3 — خريطة الأهمية المكانية لكل فريم'
    )
except Exception as e:
    print(f'لم يتم عرض الخريطة: {e}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# تطور أهمية الفريمات عبر عينات متعددة
# Aggregate frame importance over multiple samples
# ══════════════════════════════════════════════════════════════

def compute_average_frame_importance(model, data_loader, num_frames, device, max_batches=20):
    """متوسط أهمية الفريمات عبر عينات متعددة"""
    all_importances = []

    for batch_idx, batch in enumerate(data_loader):
        if batch_idx >= max_batches:
            break
        resized_imgs = batch[0][:1]   # خذ فقط أول عينة من كل batch
        try:
            imp = compute_frame_importance(model, resized_imgs, num_frames, device)
            all_importances.append(imp)
        except Exception:
            continue

    if not all_importances:
        return np.ones(num_frames) / num_frames

    return np.mean(all_importances, axis=0)


# تحميل data loader
train_loader_p3, _, _ = create_train_val_dataloader(cfg_p3)

print('⏳ حساب متوسط الأهمية عبر 20 batch...')
avg_imp = compute_average_frame_importance(model_p3, train_loader_p3, NUM_FRAMES, cfg_p3.device)

# رسم مقارنة: عينة واحدة vs متوسط
half = (NUM_FRAMES - 1) // 2
x = np.arange(NUM_FRAMES)
x_labels = [f'F{i-half:+d}' for i in range(NUM_FRAMES)]
x_labels[half] = 'F0\n(حدث)'
x_labels[-1] = f'F+{half}\n(كرة)'

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

colors = ['#A8DADC'] * NUM_FRAMES
colors[half] = '#FF6B35'
colors[NUM_FRAMES - 1] = '#E63946'

ax1.bar(x, imp_p3, color=colors, edgecolor='white')
ax1.set_xticks(x); ax1.set_xticklabels(x_labels, fontsize=8)
ax1.set_title('عينة واحدة (Single Sample)', fontweight='bold')
ax1.set_ylabel('الأهمية النسبية')
ax1.grid(axis='y', alpha=0.3)

ax2.bar(x, avg_imp, color=colors, edgecolor='white')
ax2.set_xticks(x); ax2.set_xticklabels(x_labels, fontsize=8)
ax2.set_title('متوسط 20 batch (Averaged)', fontweight='bold')
ax2.set_ylabel('الأهمية النسبية')
ax2.grid(axis='y', alpha=0.3)

for ax in (ax1, ax2):
    ax.axvline(half, color='#FF6B35', linestyle='--', alpha=0.6)
    ax.axvline(NUM_FRAMES - 1, color='#E63946', linestyle='--', alpha=0.6)

fig.suptitle('أهمية الفريمات — النموذج النهائي (Phase 3)', fontsize=14)
plt.tight_layout()
plt.show()

# أكثر الفريمات أهمية
sorted_frames = np.argsort(avg_imp)[::-1]
print('\n🏆 ترتيب الفريمات حسب الأهمية (متوسط):')
for rank, fi in enumerate(sorted_frames[:5], 1):
    rel = fi - half
    marker = ' ⚡حدث' if fi == half else (' 🏓كرة' if fi == NUM_FRAMES-1 else '')
    print(f'  #{rank}: F{rel:+d}  ({avg_imp[fi]:.4f}){marker}')

## 🔟 التقييم النهائي | Final Evaluation

In [ ]:
# ══════════════════════════════════════════════════════════════
# رسم منحنيات الخسارة من ملفات الـ log
# Plot loss curves from log files
# ══════════════════════════════════════════════════════════════

import re
import glob

def parse_loss_from_log(log_dir, phase_name):
    """استخراج قيم الخسارة من ملفات الـ log"""
    log_files = sorted(glob.glob(os.path.join(log_dir, phase_name, '*.txt')))
    if not log_files:
        return []

    train_losses = []
    pattern = re.compile(r'Epoch.*?(\d+)/.*?Loss.*?([0-9.]+(?:e[+-]?\d+)?)', re.IGNORECASE)

    for log_file in log_files:
        try:
            with open(log_file) as f:
                text = f.read()
            matches = pattern.findall(text)
            for ep, loss in matches:
                train_losses.append((int(ep), float(loss)))
        except Exception:
            continue

    return sorted(set(train_losses))


fig, axes = plt.subplots(1, 3, figsize=(18, 5))
phases = ['ttnet_1st_phase', 'ttnet_2nd_phase', 'ttnet_3rd_phase']
phase_labels = ['Phase 1: Global + Seg', 'Phase 2: Local + Event', 'Phase 3: Fine-tune']
colors_ph = ['#2196F3', '#4CAF50', '#FF5722']

any_data = False
for ax, phase, label, color in zip(axes, phases, phase_labels, colors_ph):
    log_path = os.path.join(LOGS_DIR, phase)
    losses = parse_loss_from_log(LOGS_DIR, phase)

    if losses:
        epochs, vals = zip(*losses)
        ax.plot(epochs, vals, color=color, lw=2, marker='o', markersize=4)
        ax.set_title(label, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Training Loss')
        ax.grid(alpha=0.3)
        any_data = True
    else:
        ax.text(0.5, 0.5, 'لا توجد بيانات بعد\n(سيظهر بعد التدريب)',
                ha='center', va='center', transform=ax.transAxes, fontsize=11, color='gray')
        ax.set_title(label)

fig.suptitle('منحنيات الخسارة | Training Loss Curves', fontsize=14)
plt.tight_layout()
plt.show()

if not any_data:
    print('ℹ️  منحنيات الخسارة ستظهر تلقائياً بعد إتمام التدريب')

In [ ]:
# فتح TensorBoard لمتابعة التدريب بشكل تفاعلي
# Launch TensorBoard for interactive monitoring

try:
    from tensorboard import notebook as tb_notebook
    tb_log_dir = LOGS_DIR
    print(f'🚀 فتح TensorBoard على: {tb_log_dir}')
    tb_notebook.start('--logdir', tb_log_dir)
    tb_notebook.display(height=600)
except ImportError:
    print('لفتح TensorBoard في terminal:')
    print(f'  tensorboard --logdir "{LOGS_DIR}"')
except Exception as e:
    print(f'لفتح TensorBoard في terminal:')
    print(f'  tensorboard --logdir "{LOGS_DIR}"')

In [ ]:
# ══════════════════════════════════════════════════════════════
# التقييم النهائي على مجموعة الاختبار
# Final evaluation on test set
# ══════════════════════════════════════════════════════════════

if os.path.isfile(phase3_ckpt):
    print('🔍 تشغيل التقييم على مجموعة الاختبار...')

    cmd = [
        sys.executable, 'test.py',
        '--working_dir', WORKING_DIR,
        '--saved_fn', 'ttnet_3rd_phase',
        '--batch_size', '1',
        '--pretrained_path', phase3_ckpt,
        '--seg_thresh', '0.5',
        '--event_thresh', '0.5',
        '--smooth-labelling',
    ]

    if GPU_IDX is not None:
        cmd += ['--gpu_idx', str(GPU_IDX)]
    else:
        cmd += ['--no_cuda']

    proc = subprocess.Popen(cmd, cwd=SRC_DIR,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    proc.wait()

    if proc.returncode == 0:
        print('\n✅ التقييم اكتمل')
    else:
        print(f'\n❌ خطأ في التقييم (exit code {proc.returncode})')
else:
    print('⚠️  يجب إتمام المرحلة الثالثة أولاً للتقييم')

## 📋 ملخص النتائج | Results Summary

### ما تعلمناه عن الفريمات

| الملاحظة | السبب |
|---|---|
| **F+7 (آخر فريم)** يحظى بأهمية عالية في Phase 1 | موضع الكرة مُعلَّم على آخر فريم فقط |
| **F0 (الفريم المركزي)** يكتسب أهمية في Phase 2/3 | Event spotting يعتمد على لحظة الحدث |
| الفريمات القريبة من الحدث (F±2) أهم من البعيدة | الحركة المباشرة قبل/بعد الحدث أكثر إفادة |

### التغييرات التي طبقناها
1. ✅ **رفع num_frames_sequence**: من 9 → **15** فريم (اقتراب من الـ 25 المذكورة في الورقة)
2. ✅ **إصلاح bug في TTNet.py**: `repeats=9` → `repeats=num_frames_sequence`
3. ✅ **إصلاح تحذيرات NumPy**: `np.int` → `int`، `np.float` → `float`
4. ✅ **تدريب 3 مراحل**: تدريج curriculum learning
5. ✅ **تحليل الفريمات**: Gradient saliency + spatial heatmaps